In [0]:
%pip install geopandas shapely pyproj --quiet
dbutils.library.restartPython()

In [0]:
import pandas as pd
from shapely import wkt
from shapely.geometry import Point
from pyproj import Transformer, Geod
from datetime import datetime, timezone

In [0]:
CATALOG = "gobierno_abierto"
SCHEMA_BRONZE = "a_bronze"
SCHEMA_SILVER = "b_silver"

In [0]:
CBA_LAT_RANGE = (-32.0, -30.8)
CBA_LON_RANGE = (-64.6, -63.7)

In [0]:
## 1. Paradas (stops) — limpieza

stops_bronze = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.gtfs_stops").toPandas()

print("Columnas originales:", list(stops_bronze.columns))
print(f"Filas originales: {len(stops_bronze)}")

stops = stops_bronze.copy()
stops["stop_lat"] = pd.to_numeric(stops["stop_lat"], errors="coerce")
stops["stop_lon"] = pd.to_numeric(stops["stop_lon"], errors="coerce")
stops["stop_name"] = stops["stop_name"].astype(str).str.strip()

antes = len(stops)
stops = stops.dropna(subset=["stop_id", "stop_lat", "stop_lon"])
stops = stops[(stops["stop_lat"].between(-90, 90)) & (stops["stop_lon"].between(-180, 180))]
print(f"Filas descartadas por lat/lon inválido o nulo: {antes - len(stops)}")

antes = len(stops)
stops = stops.drop_duplicates(subset=["stop_id"], keep="first")
print(f"Duplicados eliminados por stop_id: {antes - len(stops)}")

stops["geometry_wkt"] = stops.apply(lambda r: Point(r["stop_lon"], r["stop_lat"]).wkt, axis=1)

stops_silver = stops[["stop_id", "stop_name", "stop_lat", "stop_lon", "geometry_wkt"]].copy()
stops_silver["_processed_at"] = datetime.now(timezone.utc)

print(f"Filas finales: {len(stops_silver)}")
stops_silver.head()


In [0]:
## 2. Barrios — inspección de columnas

barrios_bronze = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.barrios_raw").toPandas()

print("Columnas originales:", list(barrios_bronze.columns))
print(f"Filas originales: {len(barrios_bronze)}")
barrios_bronze.head()

In [0]:
# Diagnóstico: ver qué valores de wkt_col no son geometrías válidas
from shapely import wkt as _wkt

def es_wkt_valido(valor):
    if pd.isna(valor):
        return False
    try:
        _wkt.loads(str(valor).strip())
        return True
    except Exception:
        return False

mask_invalido = ~barrios[wkt_col].apply(es_wkt_valido)
print(f"Filas con geometría inválida (no nula pero no parseable): {mask_invalido.sum()}")
display(barrios[mask_invalido])

fila_problema = barrios[mask_invalido].iloc[0]
valor = str(fila_problema[wkt_col])
print(f"Longitud del string: {len(valor)}")
print(f"Últimos 200 caracteres:\n{valor[-200:]}")
print(f"Paréntesis abiertos: {valor.count('(')} | cerrados: {valor.count(')')}")

antes = len(barrios)
barrios_invalidos = barrios[~barrios[wkt_col].apply(es_wkt_valido)]
if len(barrios_invalidos) > 0:
    print("Barrios descartados por geometría inválida/truncada en la fuente:")
    display(barrios_invalidos[["Nombre"]])

barrios_bronze = barrios[barrios[wkt_col].apply(es_wkt_valido)]
print(f"Filas descartadas por geometría inválida: {antes - len(barrios)}")

In [0]:
## 3. Barrios — detección automática de la columna de geometría
wkt_col = None
for candidato in ["geometry_wkt", "wkt", "geom", "the_geom", "geometria"]:
    if candidato in barrios_bronze.columns:
        wkt_col = candidato
        break

if wkt_col is None:
    for col in barrios_bronze.columns:
        muestra = barrios_bronze[col].dropna().astype(str).head(1)
        if len(muestra) and muestra.iloc[0].strip().upper().startswith(("POLYGON", "MULTIPOLYGON")):
            wkt_col = col
            break

if wkt_col is None:
    raise ValueError(
        f"No se pudo detectar automáticamente la columna de geometría. "
        f"Columnas disponibles: {list(barrios_bronze.columns)}. "
        f"Revisá el nombre y ajustá 'wkt_col' manualmente."
    )

print(f"Columna de geometría detectada: '{wkt_col}'")

In [0]:
## 4. Barrios — detección automática del sistema de coordenadas (CRS)

candidatos_crs = [
    "EPSG:5346",   # POSGAR 2007 / Argentina Faja 4
    "EPSG:22174",  # POSGAR 1998 / Argentina Faja 4
    "EPSG:22194",  # Campo Inchauspe / Argentina Faja 4
    "EPSG:5347",   # POSGAR 2007 / Argentina Faja 5
    "EPSG:3857",   # Web Mercator (por las dudas)
]

primer_geom = wkt.loads(barrios_bronze[wkt_col].dropna().iloc[0])
punto_muestra = primer_geom.centroid  # (x, y) en el CRS original

crs_detectado = None
for crs in candidatos_crs:
    try:
        transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
        lon, lat = transformer.transform(punto_muestra.x, punto_muestra.y)
        dentro_lat = CBA_LAT_RANGE[0] <= lat <= CBA_LAT_RANGE[1]
        dentro_lon = CBA_LON_RANGE[0] <= lon <= CBA_LON_RANGE[1]
        print(f"{crs}: lat={lat:.4f}, lon={lon:.4f} -> {'OK' if dentro_lat and dentro_lon else 'fuera de rango'}")
        if dentro_lat and dentro_lon and crs_detectado is None:
            crs_detectado = crs
    except Exception as e:
        print(f"{crs}: error al transformar ({e})")

if crs_detectado is None:
    raise ValueError(
        "Ningún CRS candidato dio un resultado dentro del área de Córdoba. "
        "Revisá manualmente las coordenadas originales y agregá el EPSG correcto "
        "a la lista de candidatos."
    )

print(f"\nCRS detectado: {crs_detectado}")

In [0]:
## 5. Barrios — reproyección, limpieza y cálculo de área

transformer = Transformer.from_crs(crs_detectado, "EPSG:4326", always_xy=True)
geod = Geod(ellps="WGS84")


def reproyectar_y_medir(wkt_str):
    """Convierte un WKT del CRS original a WGS84 y calcula el área geodésica en km²."""
    geom_original = wkt.loads(wkt_str)
    geom_wgs84 = transform_geom(geom_original, transformer)
    area_m2, _ = geod.geometry_area_perimeter(geom_wgs84)
    area_km2 = abs(area_m2) / 1_000_000
    return geom_wgs84.wkt, area_km2


def transform_geom(geom, transformer):
    from shapely.ops import transform as shapely_transform
    return shapely_transform(lambda x, y: transformer.transform(x, y), geom)


barrios = barrios_bronze.copy()

antes = len(barrios)
barrios = barrios.dropna(subset=[wkt_col])
print(f"Filas descartadas por geometría nula: {antes - len(barrios)}")

resultados = barrios[wkt_col].apply(reproyectar_y_medir)
barrios["geometry_wkt"] = resultados.apply(lambda r: r[0])
barrios["area_km2"] = resultados.apply(lambda r: r[1])

print(f"Filas finales: {len(barrios)}")
barrios[["geometry_wkt", "area_km2"]].head()

In [0]:
## 6. Barrios — armado final de la tabla Silver

def buscar_columna(columnas, candidatos):
    columnas_lower = {c.lower(): c for c in columnas}
    for candidato in candidatos:
        if candidato.lower() in columnas_lower:
            return columnas_lower[candidato.lower()]
    return None

col_nombre = buscar_columna(barrios.columns, ["nombre", "nombre_barrio", "barrio"])
col_id = buscar_columna(barrios.columns, ["barrio_id", "id", "id_barrio"])
col_tipo = buscar_columna(barrios.columns, ["tipo_barrio", "tipobarrio", "tipo"])

print(f"Columna nombre detectada: {col_nombre}")
print(f"Columna id detectada: {col_id}")
print(f"Columna tipo detectada: {col_tipo}")

barrios_silver = pd.DataFrame()
barrios_silver["barrio_id"] = barrios[col_id].astype(str) if col_id else barrios.index.astype(str)
barrios_silver["nombre"] = barrios[col_nombre].astype(str).str.strip() if col_nombre else None
barrios_silver["tipo_barrio"] = barrios[col_tipo].astype(str).str.strip() if col_tipo else None
barrios_silver["area_km2"] = barrios["area_km2"]
barrios_silver["geometry_wkt"] = barrios["geometry_wkt"]
barrios_silver["_processed_at"] = datetime.now(timezone.utc)

barrios_silver.head()

In [0]:
## 7. Escritura a Delta

spark.createDataFrame(stops_silver).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG}.{SCHEMA_SILVER}.stops"
)
spark.createDataFrame(barrios_silver).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG}.{SCHEMA_SILVER}.barrios"
)

for tabla in ["stops", "barrios"]:
    count = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.{tabla}").count()
    print(f"{CATALOG}.{SCHEMA_SILVER}.{tabla}: {count} filas")